# 🔥 Level 3 (Advanced) — High-Quality Face Swap + Identity Generation

Notebook ketiga **Learn-faceswap** — versi *advanced* dengan **2 mode**. Dirancang agar **stabil di Google Colab gratis** (versi library terbaru, minim konflik):

- **🅰️ MODE A — High-Quality Face Swap**: badan/background TETAP, hanya wajah diganti. Pakai **InsightFace + `inswapper_128`** (swap AI) lalu dipertajam **GFPGAN**.
- **🅱️ MODE B — Prompt-driven Identity Generation**: kasih 1 foto wajah + *prompt*, AI bikin gambar BARU dengan kemiripan wajah itu. Pakai **IP-Adapter (bawaan `diffusers`)** di atas **Stable Diffusion XL fotorealistik (1024px)** — stabil & hasil jauh lebih detail.

---
## ⚠️ WAJIB DIBACA DULU
1. **Aktifkan GPU**: `Runtime → Change runtime type → T4 GPU` (khusus Mode B wajib GPU).
2. **Jalankan SATU mode per sesi.** Setelah Mode A selesai, `Runtime → Restart session` sebelum Mode B.
3. **Etika & legal**: gunakan **foto milikmu / berizin**. Jangan untuk meniru identitas orang lain secara menyesatkan.
4. Jika sebuah cell error, **salin teks error-nya** — hampir semua masalah Colab berasal dari versi library yang berubah dan bisa diperbaiki cepat.

---

# 🅰️ MODE A — High-Quality Face Swap

Wajah dari **FACE** dipasang ke wajah di **BASE**; badan & background BASE tetap utuh, lalu dipertajam.

> Catatan: Mode A sengaja memakai **onnxruntime CPU** agar bebas dari error driver CUDA. Untuk 1 gambar, kecepatannya tetap beberapa detik saja.

In [ ]:
# === MODE A | Langkah 1: install + unduh model swap ===
!pip install -q insightface onnxruntime opencv-python matplotlib

import os, urllib.request
os.makedirs('models', exist_ok=True)

# Helper unduh ber-User-Agent (sebagian server menolak request tanpa UA).
def download(url, path):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r, open(path, 'wb') as f:
        f.write(r.read())
    return path

INSWAPPER = 'models/inswapper_128.onnx'
# Buang file korup/terlalu kecil (mis. hasil unduhan gagal), lalu unduh ulang.
if os.path.exists(INSWAPPER) and os.path.getsize(INSWAPPER) < 100 * 1024 * 1024:
    os.remove(INSWAPPER)
if not os.path.exists(INSWAPPER):
    mirrors = [
        'https://huggingface.co/ezioruan/inswapper_128.onnx/resolve/main/inswapper_128.onnx',
        'https://huggingface.co/datasets/Gourieff/ReActor/resolve/main/models/inswapper_128.onnx',
    ]
    for m in mirrors:
        try:
            print('Mengunduh inswapper dari', m.split('/')[2], '...')
            download(m, INSWAPPER); break
        except Exception as e:
            print('  gagal:', e)
size_mb = os.path.getsize(INSWAPPER) / 1e6 if os.path.exists(INSWAPPER) else 0
print(f'Model swap siap: {os.path.exists(INSWAPPER)} ({size_mb:.0f} MB)')
assert size_mb > 100, 'Model swap gagal terunduh utuh. Jalankan ulang cell ini.'

In [ ]:
# === MODE A | Langkah 2: load detektor + swapper (CPU, anti-error CUDA) ===
import cv2, numpy as np, insightface
from insightface.app import FaceAnalysis
from matplotlib import pyplot as plt

PROVIDERS = ['CPUExecutionProvider']
app = FaceAnalysis(name='buffalo_l', providers=PROVIDERS)
app.prepare(ctx_id=0, det_size=(640, 640))
swapper = insightface.model_zoo.get_model(INSWAPPER, providers=PROVIDERS)
assert swapper is not None, 'Swapper gagal dimuat — ulangi Langkah 1.'

def show(img_bgr, title='', size=(7, 7)):
    plt.figure(figsize=size)
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title); plt.axis('off'); plt.show()

print('InsightFace (buffalo_l) + swapper siap.')

In [ ]:
# === MODE A | Langkah 3: siapkan gambar BASE + FACE ===
# BASE = gambar tujuan (badan/scene). FACE = wajah sumber yang ingin dipasang.
# Contoh memakai wajah AI (orang TIDAK nyata) dari thispersondoesnotexist.com (1024px, bebas izin).
base_path, face_path = 'base.jpg', 'face.jpg'
try:
    download('https://thispersondoesnotexist.com/', base_path)
    download('https://thispersondoesnotexist.com/', face_path)
    print('Berhasil mengunduh 2 wajah contoh (AI).')
except Exception as e:
    print('Gagal unduh contoh:', e, '\n>> Pakai gambar sendiri lewat Opsi upload di bawah.')

# --- Pakai foto sendiri? Hapus tanda # di bawah ---
# from google.colab import files
# print('Upload BASE:');  base_path = list(files.upload().keys())[0]
# print('Upload FACE:');  face_path = list(files.upload().keys())[0]

img_base = cv2.imread(base_path)
img_face = cv2.imread(face_path)
assert img_base is not None and img_face is not None, 'Gambar gagal dibaca. Cek unduhan / upload.'
fig, ax = plt.subplots(1, 2, figsize=(11, 5))
ax[0].imshow(cv2.cvtColor(img_base, cv2.COLOR_BGR2RGB)); ax[0].set_title('BASE (tujuan)'); ax[0].axis('off')
ax[1].imshow(cv2.cvtColor(img_face, cv2.COLOR_BGR2RGB)); ax[1].set_title('FACE (wajah sumber)'); ax[1].axis('off')
plt.show()

In [ ]:
# === MODE A | Langkah 4: lakukan SWAP ===
def swap_all_faces(base_bgr, face_bgr):
    src_faces = app.get(face_bgr)
    if not src_faces:
        raise ValueError('Wajah sumber (FACE) tidak terdeteksi. Pakai foto wajah yang lebih jelas/frontal.')
    src = sorted(src_faces, key=lambda f: (f.bbox[2]-f.bbox[0])*(f.bbox[3]-f.bbox[1]))[-1]
    targets = app.get(base_bgr)
    if not targets:
        raise ValueError('Wajah di gambar BASE tidak terdeteksi.')
    out = base_bgr.copy()
    for t in targets:
        out = swapper.get(out, t, src, paste_back=True)
    return out

swapped = swap_all_faces(img_base, img_face)
show(swapped, 'HASIL SWAP (mentah, sebelum dipertajam)')

In [ ]:
# === MODE A | Langkah 5 (opsional, disarankan): pertajam wajah + upscale 2x (GFPGAN) ===
# Jika cell ini error karena dependensi, swap di Langkah 4 tetap valid — boleh dilewati.
!pip install -q gfpgan basicsr facexlib

# Perbaikan kompatibilitas: basicsr memakai modul torchvision yang sudah dihapus di versi baru
import os, basicsr
deg = os.path.join(os.path.dirname(basicsr.__file__), 'data', 'degradations.py')
src = open(deg).read().replace('torchvision.transforms.functional_tensor',
                               'torchvision.transforms.functional')
open(deg, 'w').write(src)

from gfpgan import GFPGANer
restorer = GFPGANer(
    model_path='https://github.com/TencentARC/GFPGAN/releases/download/v1.3.4/GFPGANv1.4.pth',
    upscale=2, arch='clean', channel_multiplier=2, bg_upsampler=None)
_, _, enhanced = restorer.enhance(swapped, has_aligned=False, only_center_face=False, paste_back=True)
show(enhanced, 'HASIL SWAP + GFPGAN (lebih tajam, resolusi 2x)')
cv2.imwrite('hasil_swap_mode_a.png', enhanced)
print('Tersimpan: hasil_swap_mode_a.png')

In [ ]:
# === MODE A | Langkah 6: bandingkan ===
hasil = enhanced if 'enhanced' in dir() else swapped
fig, ax = plt.subplots(1, 3, figsize=(16, 6))
for a, im, t in zip(ax, [img_base, img_face, hasil], ['BASE asli', 'FACE sumber', 'HASIL akhir']):
    a.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB)); a.set_title(t); a.axis('off')
plt.show()

---
# 🅱️ MODE B — Prompt-driven Identity Generation (IP-Adapter)

> 🔁 **`Runtime → Restart session` dulu**, lalu jalankan cell-cell di bawah dari sini. Pastikan **GPU aktif**.
> 
> ⚠️ **Alur penting:** jalankan **Langkah 1** (install) → **Restart session lagi** → baru jalankan **Langkah 2 dst**. Restart sesudah install itu wajib agar `diffusers` versi baru benar-benar terpakai (menghindari error import seperti `StableDiffusionLoraLoaderMixin`).

Konsep: kasih **1 foto wajah** + **prompt teks**, AI membuat **gambar baru** sesuai prompt dengan **kemiripan wajah** dipertahankan.

Kita pakai **IP-Adapter bawaan `diffusers`** (model `ip-adapter-plus-face`) di atas **SDXL fotorealistik (RealVisXL, 1024px)**. Pendekatan ini **stabil** di Colab (tanpa pipeline manual / versi di-pin) dan hasilnya jauh lebih detail daripada SD 1.5.

In [ ]:
# === MODE B | Langkah 1: install BERSIH diffusers (perbaikan error import) ===
# Error 'StableDiffusionLoraLoaderMixin' = instalasi diffusers campur/rusak
# (sisa versi lama bercampur versi baru). Solusi: uninstall lalu pasang ulang.
!pip uninstall -y diffusers 2>/dev/null
!pip install -q -U diffusers transformers accelerate peft safetensors
# Samakan huggingface_hub agar konsisten dgn diffusers (mengatasi error import mixin)
!pip install -q -U --force-reinstall huggingface_hub

print('\n' + '=' * 52)
print('  ✅ Selesai!  PENTING:  Runtime  ->  Restart session  SEKARANG,')
print('  lalu LANJUT ke Langkah 2 (JANGAN run ulang cell ini).')
print('=' * 52)

In [ ]:
# === MODE B | Langkah 2: load SDXL FOTOREALISTIK + IP-Adapter  (SETELAH RESTART) ===
# Hasil JAUH lebih detail & realistis (1024px) drpd SD1.5, tapi lebih lambat
# (~1 menit/gambar di T4 gratis). Ini upgrade kualitas utamanya.
import torch
assert torch.cuda.is_available(), 'GPU tidak aktif! Runtime -> Change runtime type -> T4 GPU.'
from diffusers import AutoPipelineForText2Image, DPMSolverMultistepScheduler, AutoencoderKL

# Model SDXL fotorealistik. Alternatif: 'RunDiffusion/Juggernaut-XL-v9'
# atau 'stabilityai/stable-diffusion-xl-base-1.0'.
BASE_MODEL = 'SG161222/RealVisXL_V4.0'

# VAE khusus fp16 -> mencegah gambar hitam/NaN yang umum pada SDXL fp16.
vae = AutoencoderKL.from_pretrained('madebyollin/sdxl-vae-fp16-fix', torch_dtype=torch.float16)
pipe = AutoPipelineForText2Image.from_pretrained(
    BASE_MODEL, vae=vae, torch_dtype=torch.float16, variant='fp16', use_safetensors=True)

# Sampler DPM++ 2M Karras = detail lebih halus
pipe.scheduler = DPMSolverMultistepScheduler.from_config(
    pipe.scheduler.config, use_karras_sigmas=True, algorithm_type='dpmsolver++')

# IP-Adapter SDXL 'plus-face' = transfer ciri wajah dari gambar referensi
pipe.load_ip_adapter('h94/IP-Adapter', subfolder='sdxl_models',
                     weight_name='ip-adapter-plus-face_sdxl_vit-h.safetensors')
pipe.set_ip_adapter_scale(0.7)        # 0-1: makin tinggi = wajah makin mirip referensi
pipe.enable_model_cpu_offload()       # muat di T4 16GB (jangan pakai .to('cuda') bersamaan)
print('Pipeline SDXL + IP-Adapter siap.')

In [ ]:
# === MODE B | Langkah 3: siapkan wajah referensi (UPLOAD atau contoh) ===
import os, urllib.request
from PIL import Image
from IPython.display import display

# >>> PILIH SUMBER WAJAH:
#     'upload' = unggah foto wajahmu sendiri (muncul tombol Choose Files)
#     'contoh' = pakai wajah AI otomatis (orang tidak nyata)
SUMBER_WAJAH = 'upload'

def _dl(url, path):
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=180) as r, open(path, 'wb') as f:
        f.write(r.read())

if SUMBER_WAJAH == 'upload':
    from google.colab import files
    print('Klik tombol di bawah, lalu pilih file foto wajah (jpg/png)...')
    up = files.upload()                     # <-- tombol upload muncul di sini
    face_ref_path = list(up.keys())[0]
else:
    face_ref_path = 'face.jpg'
    if not os.path.exists(face_ref_path):
        _dl('https://thispersondoesnotexist.com/', face_ref_path)  # wajah AI

face_image = Image.open(face_ref_path).convert('RGB').resize((512, 512))
print('Wajah referensi (input):', face_ref_path)
display(face_image)

In [ ]:
# === MODE B | Langkah 4: GENERATE gambar baru dari prompt (HQ 1024px) ===
from IPython.display import display

prompt = 'cinematic photo of a person as an astronaut, detailed space suit, dramatic lighting, ultra realistic, highly detailed, sharp focus, 8k'
negative = 'blurry, low quality, low resolution, distorted, deformed, extra fingers, cartoon, painting, text, watermark, jpeg artifacts'

image = pipe(
    prompt=prompt,
    negative_prompt=negative,
    ip_adapter_image=face_image,        # wajah referensi
    width=1024, height=1024,            # resolusi tinggi (SDXL)
    num_inference_steps=30,
    guidance_scale=5.0,
    generator=torch.Generator('cpu').manual_seed(42),   # ganti angka seed -> variasi baru
).images[0]

image.save('hasil_modeb.png')
print('Tersimpan: hasil_modeb.png (1024x1024)')
display(image)

# (opsional) unduh hasil ke komputermu — hapus tanda # di baris bawah:
# from google.colab import files; files.download('hasil_modeb.png')

## 🎛️ Cara mengatur hasil (Mode B)

| Parameter | Fungsi | Tips |
|-----------|--------|------|
| `prompt` | Deskripsi gambar | Makin detail makin terarah |
| `set_ip_adapter_scale` | Kekuatan kemiripan wajah | 0.6-0.9; naikkan bila kurang mirip |
| `manual_seed` | Angka acak awal | Seed sama = hasil sama; ganti = variasi |
| `num_inference_steps` | Langkah denoising | 25-40 |
| `guidance_scale` | Kepatuhan ke prompt | 4-6 untuk SDXL |

**Ingin lebih bagus lagi?**
- **Kemiripan wajah lebih kuat**: ganti bobot ke **`ip-adapter-faceid`** (butuh `insightface` untuk embedding), atau untuk identitas terkuat pakai **InstantID** (lebih berat & rewel versi).
- **Lebih tajam/HD**: jalankan hasil lewat **upscaler** (Real-ESRGAN) atau **CodeFormer/GFPGAN** untuk mempertajam wajah (seperti enhancement di Mode A).
- **Variasi cepat**: ganti angka `manual_seed`, atau coba model lain (`RunDiffusion/Juggernaut-XL-v9`).
- **Kalau lambat di T4**: turunkan `width/height` ke 768, atau kurangi `num_inference_steps` ke 25.

---
## 🧠 Ringkasan & Perbandingan

| | Mode A (Swap) | Mode B (IP-Adapter) |
|---|---|---|
| Input | BASE + FACE | FACE + prompt teks |
| Yang berubah | Hanya wajah | Seluruh gambar (baru) |
| Background/badan | Tetap utuh | Dibuat ulang sesuai prompt |
| Kontrol kreatif | Rendah | Tinggi (lewat prompt) |
| Kebutuhan | Ringan (CPU ok) | GPU (T4 cukup) |

### 🔧 Tool/alternatif terbaik lain (open-source)
- **FaceFusion** — paling turnkey untuk swap foto & **video** (ada enhancer built-in)
- **IP-Adapter FaceID / InstantID** — identitas lebih kuat untuk Mode B
- **CodeFormer** — alternatif GFPGAN untuk restorasi wajah
- **Real-ESRGAN** — upscaler resolusi keseluruhan gambar

### 💡 Kunci hasil bagus
1. Foto wajah referensi **tajam, terang, frontal, besar di frame**
2. Selalu **pertajam** (GFPGAN/CodeFormer) setelah swap
3. Untuk Mode B: eksperimen `seed`, `ip_adapter_scale`, dan prompt

### ⚠️ Pengingat etika
Gunakan hanya foto milikmu / yang berizin. Jangan untuk meniru orang lain secara menyesatkan.